In [ ]:
"""
MOBILENET-UNET FOR DRISHTI-GS DATASET
=====================================
Adapted for Drishti-GS dataset structure:
- Merges Training/Testing folders
- Performs random split
- Handles Drishti-specific resolution and mask formats
"""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import albumentations as A
from PIL import Image

# ============================================================================
# GPU CONFIGURATION
# ============================================================================
def configure_gpu():
    """Configure GPU settings for optimal performance"""
    print("Configuring GPU settings...")
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        try:
            for device in physical_devices:
                tf.config.experimental.set_memory_growth(device, True)
            print(f"✓ GPU configured: {len(physical_devices)} device(s) available")
            return True
        except RuntimeError as e:
            print(f"GPU configuration error: {e}")
            return False
    else:
        print("⚠ No GPU detected, running on CPU")
        return False

# ============================================================================
# DRISHTI-GS DATA LOADING
# ============================================================================

def find_mask_for_image(img_path):
    """
    Attempt to find corresponding mask for Drishti-GS image.
    Strategies:
    1. Look for '*_cupsegSoftmap.png' and '*_ODsegSoftmap.png' (Standard Drishti)
    2. Look for 'GT' folder in parent directories.
    """
    img_dir = os.path.dirname(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    
    # Strategy 1: Standard Drishti softmaps in same or 'Test_GT'/'Training_GT' folders
    # Usually Drishti has: 'drishtiGS_001.png' -> 'drishtiGS_001_cupsegSoftmap.png'
    
    # Try to find cup and disc masks
    # Common locations for GT in Drishti structure
    search_paths = [
        img_dir,
        os.path.join(os.path.dirname(img_dir), 'GT'),
        os.path.join(img_dir, '..', 'GT'),  # Go up one level
        os.path.join(img_dir, '..', '..', 'GT') # Go up two levels
    ]
    
    cup_path = None
    disc_path = None
    
    # Recursively search for matching ID files
    for root, _, files in os.walk(os.path.dirname(os.path.dirname(img_dir))): # Search broadly nearby
        for f in files:
            if img_name in f:
                if 'cup' in f.lower() and ('seg' in f.lower() or 'map' in f.lower()):
                    cup_path = os.path.join(root, f)
                if ('od' in f.lower() or 'disc' in f.lower()) and ('seg' in f.lower() or 'map' in f.lower()):
                    disc_path = os.path.join(root, f)
                    
    return cup_path, disc_path

def process_drishti_mask(cup_path, disc_path, target_size=(512,512)):
    """Combine separate Cup and Disc masks into single 3-class mask"""
    if not cup_path or not disc_path:
        return None
        
    try:
        # Load softmaps (typically grayscale, values indicate confidence)
        cup = cv2.imread(cup_path, cv2.IMREAD_GRAYSCALE)
        disc = cv2.imread(disc_path, cv2.IMREAD_GRAYSCALE)
        
        if cup is None or disc is None:
            return None
            
        cup = cv2.resize(cup, target_size)
        disc = cv2.resize(disc, target_size)
        
        # Threshold to get binary masks (Drishti softmaps)
        # Usually > 0 or > 127 is foreground
        _, cup_bin = cv2.threshold(cup, 127, 255, cv2.THRESH_BINARY)
        _, disc_bin = cv2.threshold(disc, 127, 255, cv2.THRESH_BINARY)
        
        # Create combined mask: 0=Bg, 1=Disc, 2=Cup
        # Initialize with Background
        mask = np.zeros(target_size, dtype=np.uint8)
        
        # Set Disc (1)
        mask[disc_bin > 0] = 1
        
        # Set Cup (2) - Cup overlaps Disc, so it takes precedence
        mask[cup_bin > 0] = 2
        
        return mask
    except Exception as e:
        print(f"Error processing masks {cup_path}, {disc_path}: {e}")
        return None

def load_drishti_data(root_dir, img_size=(512,512)):
    """
    Load Drishti-GS data from provided root directory.
    Recursively finds all images in 'glaucoma' and 'normal' subfolders.
    """
    print(f"Scanning for data in: {root_dir}")
    
    image_extensions = ['*.png', '*.jpg', '*.jpeg']
    all_image_paths = []
    
    # Walk through the directory to find all images
    for root, dirs, files in os.walk(root_dir):
        for ext in image_extensions:
            all_image_paths.extend(glob.glob(os.path.join(root, ext)))
            
    # Filter out mask files if they are mixed in (often contain 'seg', 'map', 'GT')
    clean_image_paths = [p for p in all_image_paths if 'seg' not in p.lower() and 'map' not in p.lower() and 'gt' not in p.lower()]
    
    print(f"Found {len(clean_image_paths)} potential images.")
    
    images = []
    masks = []
    valid_count = 0
    
    for i, img_path in enumerate(clean_image_paths):
        # Load Image
        try:
            img = cv2.imread(img_path)
            if img is None: continue
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, img_size) / 255.0
            
            # Find and Load Mask
            cup_path, disc_path = find_mask_for_image(img_path)
            
            if cup_path and disc_path:
                mask = process_drishti_mask(cup_path, disc_path, img_size)
                if mask is not None:
                    images.append(img)
                    masks.append(mask)
                    valid_count += 1
            else:
                # Fallback: Check if there's a single mask file (rare for Drishti but possible in processed datasets)
                # Or skip if no GT found
                if i < 5: print(f"  No masks found for: {os.path.basename(img_path)}")
                pass
                
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            
        if (i+1) % 20 == 0:
            print(f"Processed {i+1} files...")
            
    print(f"Successfully loaded {valid_count} paired images and masks.")
    
    if valid_count == 0:
        print("WARNING: No valid image-mask pairs found. Please check dataset structure.")
        print(f"Example search root: {root_dir}")
        
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.uint8)

def create_drishti_splits(all_images, all_masks):
    """
    Merge all and create random splits.
    Drishti-GS is small (~101 images).
    We will use a 60% Train, 20% Val, 20% Test split.
    """
    print("Creating random splits (Merge Training+Testing)...")
    
    # First split: Test (20%)
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        all_images, all_masks, test_size=0.2, random_state=42
    )
    
    # Second split: Train/Val (from remaining 80%, take 25% to get 20% of total)
    # 0.25 * 0.8 = 0.2 total
    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val, y_train_val, test_size=0.25, random_state=42
    )
    
    print(f"Split Sizes - Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

# ---- Augmentation Setup ----
def setup_augmentations():
    return A.Compose([
        A.RandomBrightnessContrast(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=30, p=0.5),
        A.ElasticTransform(p=0.3) # Effective for small medical datasets
    ])

# ---- Data Generator ----
def data_generator(images, masks, batch_size, augment=None):
    idxs = np.arange(len(images))
    while True:
        np.random.shuffle(idxs)
        for i in range(0, len(images), batch_size):
            batch_idxs = idxs[i:i+batch_size]
            batch_x, batch_y = [], []
            
            for idx in batch_idxs:
                img, mask = images[idx], masks[idx]
                
                if augment:
                    aug = augment(image=img, mask=mask)
                    img_aug, mask_aug = aug['image'], aug['mask']
                else:
                    img_aug, mask_aug = img, mask
                
                mask_cat = to_categorical(mask_aug, num_classes=3)
                batch_x.append(img_aug)
                batch_y.append(mask_cat)
            
            if len(batch_x) > 0:
                yield np.stack(batch_x).astype(np.float32), np.stack(batch_y).astype(np.float32)

# ============================================================================
# MOBILENET-UNET MODEL
# ============================================================================

def build_mobilenet_unet(input_shape=(512,512,3), num_classes=3):
    """Build MobileNet-UNet"""
    inputs = layers.Input(input_shape, dtype='float32')
    backbone = MobileNetV2(input_tensor=inputs, weights='imagenet', include_top=False)
    
    # Encoder Layers
    skip1 = backbone.get_layer('block_1_expand_relu').output  # 128x128
    skip2 = backbone.get_layer('block_3_expand_relu').output  # 64x64
    skip3 = backbone.get_layer('block_6_expand_relu').output  # 32x32
    skip4 = backbone.get_layer('block_13_expand_relu').output # 16x16
    bridge = backbone.output # 8x8
    
    # Decoder
    x = layers.UpSampling2D((2, 2))(bridge)
    x = layers.Concatenate()([x, skip4])
    x = layers.Conv2D(512, (3, 3), activation='relu', padding='same')(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skip3])
    x = layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skip2])
    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Concatenate()([x, skip1])
    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
    
    x = layers.UpSampling2D((2, 2))(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    
    outputs = layers.Conv2D(num_classes, (1, 1), activation='softmax')(x)
    
    return models.Model(inputs, outputs, name="MobileNet-UNet_Drishti")

# ---- Metrics ----
def dice_coef_multiclass(y_true, y_pred, smooth=1e-7):
    y_true_f, y_pred_f = K.flatten(y_true), K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)

def dice_coef_class(y_true, y_pred, class_index, smooth=1e-7):
    y_true_c = K.cast(K.equal(K.argmax(y_true, axis=-1), class_index), 'float32')
    y_pred_c = K.cast(K.equal(K.argmax(y_pred, axis=-1), class_index), 'float32')
    intersection = K.sum(y_true_c * y_pred_c)
    return (2. * intersection + smooth) / (K.sum(y_true_c) + K.sum(y_pred_c) + smooth)

def dice_class_1(y_true, y_pred): return dice_coef_class(y_true, y_pred, 1) # Disc
def dice_class_2(y_true, y_pred): return dice_coef_class(y_true, y_pred, 2) # Cup

# ---- Training ----
def train_model(X_train, y_train, X_val, y_val, model_name, augment=None):
    K.clear_session()
    model = build_mobilenet_unet()
    
    model.compile(optimizer=optimizers.Adam(1e-4), loss="categorical_crossentropy",
                  metrics=[dice_coef_multiclass, dice_class_1, dice_class_2, "accuracy"])
    
    checkpoint = ModelCheckpoint(f"{model_name}.keras", save_best_only=True, monitor='val_loss')
    early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    
    batch_size = 4
    history = model.fit(
        data_generator(X_train, y_train, batch_size, augment),
        validation_data=data_generator(X_val, y_val, batch_size),
        steps_per_epoch=len(X_train)//batch_size,
        validation_steps=len(X_val)//batch_size,
        epochs=100, # Increased epochs for small dataset
        callbacks=[checkpoint, early_stop],
        verbose=1
    )
    return model, history

def save_segmentation_masks(X_test, y_pred_labels, save_dir='results/segmentation_masks'):
    """Save masks with visible pixel values (0, 127, 255)"""
    os.makedirs(save_dir, exist_ok=True)
    print(f"Saving {len(X_test)} masks to {save_dir}...")
    for i, pred_mask in enumerate(y_pred_labels):
        visible_mask = np.zeros_like(pred_mask, dtype=np.uint8)
        visible_mask[pred_mask == 1] = 127
        visible_mask[pred_mask == 2] = 255
        Image.fromarray(visible_mask).save(f"{save_dir}/test_{i+1:03d}_mask.png")

def evaluate_and_save(model, X_test, y_test, model_name):
    print(f"Evaluating {model_name}...")
    y_pred = model.predict(X_test, batch_size=4)
    y_pred_labels = np.argmax(y_pred, axis=-1)
    
    # Metrics calculation
    y_test_flat = np.argmax(y_test, axis=-1).flatten()
    y_pred_flat = y_pred_labels.flatten()
    
    f1 = f1_score(y_test_flat, y_pred_flat, average='macro')
    print(f"Macro F1 Score: {f1:.4f}")
    
    # Save results
    save_segmentation_masks(X_test, y_pred_labels, f'results/{model_name}_masks')
    return y_pred_labels

# ---- Main ----
if __name__ == "__main__":
    configure_gpu()
    
    # Path to Drishti-GS
    root_dir = "/kaggle/input/datasets/ayush02102001/glaucoma-classification-datasets/DRISHTI-GS/DRISHTI-GS/"
    
    # Load
    X, y = load_drishti_data(root_dir)
    
    if len(X) > 0:
        # Split (Merge -> Random Split)
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = create_drishti_splits(X, y)
        
        # Augmentation
        aug = setup_augmentations()
        
        # Train
        model_name = "MobileNetV2_DrishtiGS"
        model, history = train_model(X_train, y_train, X_val, y_val, model_name, aug)
        
        # Evaluate
        evaluate_and_save(model, X_test, y_test, model_name)
        print("Done!")
    else:
        print("Failed to load data. Please verify path and masks.")